# LLM Trip Parsing Pipeline

This notebook isolates the LLM-related logic from `charging_calendar_ui.py`. It provides functions to build prompts, call Ollama or a local Hugging Face `transformers` pipeline, and extract JSON output for trip requests. No UI, calendar rendering, or speech-to-text code is included.

In [47]:
# Imports and safe imports
import json
import re
import urllib.request
import urllib.error
from datetime import date, timedelta

try:
    from transformers import pipeline
except Exception:
    pipeline = None
    print("WARNING: transformers.pipeline not available; install transformers to use local models")

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    RecursiveCharacterTextSplitter = None
    print("WARNING: langchain_text_splitters not available; install langchain-text-splitters to use RecursiveCharacterTextSplitter")

In [48]:
system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and convert it into a strict JSON object that describes one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message asking the user for the missing details.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Required fields for each trip
- action: one of add_trip, change_trip, delete_trip
- title: the trip name implied by the user
- date: exact date in ISO format YYYY-MM-DD
- from: start location, city and/or street if known
- to: destination location, city and/or street if known
- Time_leave: departure time in HH:MM 24-hour format
- Time_arrival: arrival time in HH:MM 24-hour format

Rules
1. Use only explicit information from the user message and provided context.
2. Do not speculate about missing dates, times, locations, or trip intent.
3. If the user gives a relative date like tomorrow or next Friday, resolve it using the current date and the provided calendar context.
4. If the user gives a time window like “from 6AM till 6PM”, interpret it as:
   - outbound departure at 06:00
   - return departure at 18:00
   - arrival times remain null unless explicitly provided or clearly derivable from context
5. If the destination or return location is not explicit, use null.
6. If a time is not explicit, use null.
7. If the message contains only one trip, output one record.
8. If the message contains multiple trips, output one record per trip.
9. Make sure the returned structure is valid JSON.
10. Do not return any text outside the JSON object.
11. Time_leave is used when the user specifies he is leaving from the start location or destination.
12. Time_arrival is used when the user specifies he is arriving at the location or the final destination
13. Title needs to be the same for both outbound and return trip when it is a round trip, so that they can be linked together in the calendar.
14. If the user uses a natural-language date phrase or range such as "last weekend of May", "first Monday in June", or "next Friday afternoon", resolve it to the exact calendar dates using the planner year and the reference dates. Do not guess. For example, "last weekend of May" means the last Saturday and Sunday that occur in May of the planner year. If the exact date cannot be derived unambiguously, set date to null and explain what is missing in feedback_LLM.
Output structure
- Use numbered keys for each trip: "1", "2", "3", ...
- Include a final field named feedback_LLM.
- Example output:
  {
    "1": {
      "action": "add_trip",
      "title": "Work",
      "date": "2026-05-14",
      "from": "Gent",
      "to": "Office, Brussels",
      "Time_leave": "06:00",
      "Time_arrival": null
    },
    "2": {
      "action": "add_trip",
      "title": "Return home",
      "date": "2026-05-14",
      "from": "Office, Brussels",
      "to": "Gent",
      "Time_leave": "18:00",
      "Time_arrival": null
    },
    "feedback_LLM": "All required information is present."
  }

Important
- Use null, not guessed values.
- Never invent dates or times.
- Prefer precision over completeness.
- The output must be suitable for downstream JSON parsing.
"""

In [49]:
class LLMPipeline:
    def __init__(
        self,
        ollama_base_url: str = "http://localhost:11434",
        ollama_model: str = "llama3:8b",
        backend: str = "auto",
        display_year: int | None = None,
        home_location: str = "Gent",
    ) -> None:
        self.ollama_base_url = ollama_base_url.rstrip("/")
        self.ollama_model = ollama_model
        self.backend = (backend or "auto").lower()
        self.display_year = display_year or date.today().year
        self.home_location = home_location
        self.llm_generator = None

    def parse_trip(self, message: str, backend: str | None = None):
        backend_to_use = (backend or self.backend or "auto").lower()
        return self._interpret_with_llm(message, backend_to_use)

    def _interpret_with_llm(self, message: str, backend: str = "auto"):
        if backend == "ollama":
            return self._interpret_with_ollama(message)
        if backend == "transformers":
            return self._interpret_with_transformers(message)

        # auto: try Ollama first, fall back to transformers
        parsed, raw, status = self._interpret_with_ollama(message)
        if parsed is not None:
            return parsed, raw, status
        if status == "Ollama unavailable":
            return self._interpret_with_transformers(message)
        return None, raw, status

    # --- Ollama support ---
    def _interpret_with_ollama(self, message: str):
        today = date.today()
        today_iso = today.isoformat()
        today_weekday = today.strftime("%A")
        prompt = self._build_trip_prompt(message, today_iso, today_weekday)

        base_url = self.ollama_base_url or "http://localhost:11434"
        configured_model = self.ollama_model or "llama3:8b"

        # Resolve model (best-effort)
        model, model_note = self._resolve_ollama_model(base_url, configured_model)

        # Try endpoints
        response_text = self._call_ollama_endpoint(base_url, model, prompt, "/api/generate")
        if response_text is None:
            response_text = self._call_ollama_endpoint(base_url, model, prompt, "/api/chat")

        if response_text is None:
            return None, None, "Both Ollama endpoints failed. Check model name and Ollama version."

        try:
            response_json = json.loads(response_text)
        except json.JSONDecodeError:
            return None, response_text, "Ollama returned invalid JSON"

        generated = str(response_json.get("response", "")).strip()
        if not generated:
            return None, response_text, "Ollama did not return any text"

        parsed = self._extract_json_object(generated)
        if parsed is None:
            return None, generated, "Ollama output was unclear"

        if not self._is_multi_trip_payload(parsed) and parsed.get("action") != "add_trip":
            parsed["action"] = "add_trip"

        status = "Parsed with Ollama"
        if model_note:
            status = f"{status}. {model_note}"
        return parsed, generated, status

    def _resolve_ollama_model(self, base_url: str, configured_model: str):
        models = self._list_ollama_models(base_url)
        if not models:
            return configured_model, None
        if configured_model in models:
            return configured_model, None
        if ":" not in configured_model:
            latest_candidate = f"{configured_model}:latest"
            if latest_candidate in models:
                return latest_candidate, f"Using available model '{latest_candidate}'"
        fallback_model = models[0]
        return fallback_model, f"Model '{configured_model}' not found, using '{fallback_model}'"

    def _list_ollama_models(self, base_url: str):
        try:
            request = urllib.request.Request(f"{base_url}/api/tags", headers={"Content-Type": "application/json"}, method="GET")
            with urllib.request.urlopen(request, timeout=10) as response:
                raw = response.read().decode("utf-8", errors="replace")
            payload = json.loads(raw)
            models = payload.get("models", []) if isinstance(payload, dict) else []
            names = [str(item.get("name", "")).strip() for item in models if isinstance(item, dict)]
            return [name for name in names if name]
        except Exception:
            return []

    def _call_ollama_endpoint(self, base_url: str, model: str, prompt: str, endpoint: str):
        payload = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0.1, "num_predict": 300},
        }
        try:
            request = urllib.request.Request(
                f"{base_url}{endpoint}",
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with urllib.request.urlopen(request, timeout=150) as response:
                return response.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as exc:
            # Map some common errors to None so caller can try alternative endpoints
            if exc.code in (404, 500):
                return None
            return None
        except Exception:
            return None

    # --- Transformers support ---
    def _interpret_with_transformers(self, message: str):
        if pipeline is None:
            return None, None, "Transformers pipeline unavailable"

        if self.llm_generator is None:
            try:
                self.llm_generator = pipeline("text-generation", model="meta-llama/Llama-3.2-3B-Instruct")
            except Exception as exc:
                self.llm_generator = None
                return None, None, f"Transformers model failed to load: {exc}"

        today = date.today()
        prompt = self._build_trip_prompt(message, today.isoformat(), today.strftime("%A"))

        try:
            output = self.llm_generator(prompt, max_new_tokens=120, do_sample=False, temperature=0.1, return_full_text=False)
        except Exception as exc:
            return None, None, f"Transformers generation failed: {exc}"

        if not output:
            return None, None, "Transformers returned no output"

        generated = output[0].get("generated_text", "").strip()
        parsed = self._extract_json_object(generated)
        if parsed is None:
            return None, generated, "Transformers output was unclear"
        if not self._is_multi_trip_payload(parsed) and parsed.get("action") != "add_trip":
            parsed["action"] = "add_trip"
        return parsed, generated, "Parsed with Transformers"

    # --- Prompt building and JSON extraction ---
    def _build_trip_prompt(self, message: str, today_iso: str, today_weekday: str) -> str:
        today = date.fromisoformat(today_iso)
        reference_dates = [
            f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
            for offset in range(1, 8)
        ]
        reference_dates_text = "\n".join(reference_dates)
        display_year = int(self.display_year)
        home_location = (self.home_location or "unknown").strip() or "unknown"
        system_prompt = (
            f"{system_prompt_advanced}\n\n"
            f"Today: {today_weekday} {today_iso}. Year: {display_year}. Home location: {home_location}\n\n"
            f"Reference dates:\n{reference_dates_text}\n\n"
        )

        return f"{system_prompt}User: {message}\nJSON:"

    def _extract_json_object(self, text: str):
        try:
            maybe = json.loads(text)
            if isinstance(maybe, dict):
                return maybe
        except json.JSONDecodeError:
            pass

        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None

        candidate = text[start : end + 1]
        try:
            maybe = json.loads(candidate)
        except json.JSONDecodeError:
            return None
        if isinstance(maybe, dict):
            return maybe
        return None

    def _is_multi_trip_payload(self, payload) -> bool:
        return isinstance(payload, dict) and any(str(key).isdigit() for key in payload.keys())

Test

In [50]:
test_messages = [
    "trip back and forth to antwerp leave at 6AM return at 6PM tomorrow",
    "trip next friday to brussels from 8AM and be back home at 12AM",
    "On 25th of August I go to Koksijde and leave around 9AM and will be back home at 9PM",
    "The last weekend of May I will go on Saturday at 12 AM to Leuven and return the day after at home around 10PM",
    "weekend trip to antwerp",
]

test_pipeline = pipeline if isinstance(pipeline, LLMPipeline) else LLMPipeline(backend="ollama")

for message in test_messages:
    parsed, raw, status = test_pipeline.parse_trip(message, backend="ollama")
    print("\n---")
    print("Message:", message)
    print("Status:", status)
    print("Raw output:", raw)
    print("Parsed:", parsed)


---
Message: trip back and forth to antwerp leave at 6AM return at 6PM tomorrow
Status: Parsed with Ollama
Raw output: {
"1": {
"action": "add_trip",
"title": "Trip to Antwerp",
"date": "2026-05-14",
"from": "Gent",
"to": "Antwerp",
"Time_leave": "06:00",
"Time_arrival": null
},
"2": {
"action": "add_trip",
"title": "Return trip from Antwerp",
"date": "2026-05-14",
"from": "Antwerp",
"to": "Gent",
"Time_leave": "18:00",
"Time_arrival": null
},
"feedback_LLM": "All required information is present."
}
Parsed: {'1': {'action': 'add_trip', 'title': 'Trip to Antwerp', 'date': '2026-05-14', 'from': 'Gent', 'to': 'Antwerp', 'Time_leave': '06:00', 'Time_arrival': None}, '2': {'action': 'add_trip', 'title': 'Return trip from Antwerp', 'date': '2026-05-14', 'from': 'Antwerp', 'to': 'Gent', 'Time_leave': '18:00', 'Time_arrival': None}, 'feedback_LLM': 'All required information is present.'}

---
Message: trip next friday to brussels from 8AM and be back home at 12AM
Status: Parsed with Ollama
Ra

In [51]:
# Install chromadb and langchain if needed
import subprocess
import sys

packages_to_install = []

try:
    import chromadb
except ImportError:
    packages_to_install.append("chromadb")

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    packages_to_install.append("langchain-text-splitters")

if packages_to_install:
    print(f"Installing: {', '.join(packages_to_install)}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages_to_install)
    print("Installation complete.")
else:
    print("All packages already installed.")

All packages already installed.


In [52]:
from pathlib import Path

class RAGUserProfileManager:
    """Load user profile from markdown and store in ChromaDB for retrieval."""
    
    def __init__(self, profile_path: str, collection_name: str = "user_profile", chunk_size: int = 200, chunk_overlap: int = 50):
        self.profile_path = Path(profile_path)
        self.collection_name = collection_name
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.client = chromadb.Client()
        self.collection = None
        self.profile_sections = {}
        
    def load_profile(self):
        """Parse user_profile.md and split into chunks using RecursiveCharacterTextSplitter."""
        if not self.profile_path.exists():
            raise FileNotFoundError(f"Profile not found: {self.profile_path}")
        
        content = self.profile_path.read_text(encoding="utf-8")
        sections = self._parse_sections(content)
        self.profile_sections = sections
        print(f"Loaded {len(sections)} profile chunks.")
        return sections
    
    def _parse_sections(self, content: str) -> dict:
        """Split markdown into overlapping chunks using RecursiveCharacterTextSplitter."""
        if RecursiveCharacterTextSplitter is None:
            raise ImportError("langchain not available. Install langchain to use RecursiveCharacterTextSplitter.")
        
        # Use RecursiveCharacterTextSplitter with sliding window semantics
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", "## ", "# ", " ", ""]
        )
        
        # Split content into chunks
        chunks = splitter.split_text(content)
        
        # Create sections indexed by chunk number
        sections = {}
        for idx, chunk in enumerate(chunks):
            section_key = f"chunk_{idx}"
            sections[section_key] = chunk.strip()
        
        return sections
    
    def index_profile(self):
        """Store profile chunks in ChromaDB."""
        if not self.profile_sections:
            self.load_profile()
        
        # Create or get collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        # Add each chunk as a document
        for chunk_key, chunk_text in self.profile_sections.items():
            # Skip empty chunks
            if not chunk_text.strip():
                continue
            
            self.collection.add(
                ids=[chunk_key],
                documents=[chunk_text],
                metadatas=[{"chunk": chunk_key}]
            )
        
        print(f"Indexed {len([c for c in self.profile_sections if c.strip()])} chunks into ChromaDB.")
    
    def retrieve_context(self, query: str, top_k: int = 3) -> str:
        """Query ChromaDB and return relevant context."""
        if not self.collection:
            self.index_profile()
        
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        
        if not results or not results["documents"] or not results["documents"][0]:
            return ""
        
        # Combine retrieved documents
        context_parts = []
        for doc in results["documents"][0]:
            if doc.strip():
                context_parts.append(doc.strip())
        
        return "\n\n".join(context_parts)
    
    def get_full_profile_summary(self) -> str:
        """Return a concise summary of the user profile from chunks."""
        if not self.profile_sections:
            self.load_profile()
        
        # Collect first few chunks into summary
        summary_parts = []
        for idx in range(min(3, len(self.profile_sections))):
            key = f"chunk_{idx}"
            if key in self.profile_sections:
                text = self.profile_sections[key].strip()
                if text:
                    summary_parts.append(text[:200])  # Take first 200 chars per chunk
        
        return " ".join(summary_parts)

In [53]:
class LLMPipelineWithRAG(LLMPipeline):
    """Extended LLMPipeline that integrates RAG context from user profile."""
    
    def __init__(self, rag_manager: RAGUserProfileManager, **kwargs):
        super().__init__(**kwargs)
        self.rag_manager = rag_manager
        self.rag_context_cache = {}
    
    def _build_trip_prompt(self, message: str, today_iso: str, today_weekday: str) -> str:
        """Build prompt with RAG context injected."""
        today = date.fromisoformat(today_iso)
        today = date.today()
        today_iso = today.isoformat()
        today_weekday = today.strftime("%A")
        
        reference_dates = [
            f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
            for offset in range(1, 8)
        ]
        reference_dates_text = "\n".join(reference_dates)
        display_year = int(self.display_year)
        home_location = (self.home_location or "unknown").strip() or "unknown"
        
        # Retrieve RAG context
        rag_context_block = self.rag_manager.retrieve_context(message, top_k=3)
        
        system_prompt = (
            f"{system_prompt_advanced}\n\n"
            f"Today: {today_weekday} {today_iso}. Year: {display_year}. Home location: {home_location}\n\n"
            f"Reference dates:\n{reference_dates_text}\n\n"
            f"Relevant user profile context:\n"
            f"---\n{rag_context_block}\n---\n\n"
            f"Use the relevant user profile context when it helps resolve ambiguity, but never invent missing facts."
        )

        return f"{system_prompt}\nUser: {message}\nJSON:"

In [54]:
# Quick check: confirm RAG context is injected in prompt
preview_prompt = rag_pipeline._build_trip_prompt(
    message="schedule next friday as a normal workweek",
    today_iso=date.today().isoformat(),
    today_weekday=date.today().strftime("%A"),
)
print("Contains RAG marker:", "Relevant user profile context:" in preview_prompt)
print("\nPrompt preview (first 900 chars):\n")
print(preview_prompt[:900])

Contains RAG marker: True

Prompt preview (first 900 chars):

You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and convert it into a strict JSON object that describes one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message asking the user for the missing details.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car


In [55]:
today = date.today()
today_iso = today.isoformat()
today_weekday = today.strftime("%A")

# Define variables needed for system prompt
display_year = 2026
home_location = "Gent"

# Build reference dates (next 7 days)
reference_dates = [
    f"{(today + timedelta(days=offset)).isoformat()} {(today + timedelta(days=offset)).strftime('%A')}"
    for offset in range(1, 8)
]
reference_dates_text = "\n".join(reference_dates)

# Build system prompt
system_prompt = (
    f"{system_prompt_advanced}\n\n"
    f"Today: {today_weekday} {today_iso}. Year: {display_year}."
    f"Reference dates:\n{reference_dates_text}\n\n"
)
print(system_prompt)

You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and convert it into a strict JSON object that describes one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message asking the user for the missing details.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Requ

## Test RAG with User Profile

In [56]:
# Initialize RAG with user profile
# Adjust the path if needed - should point to your user_profile.md
profile_path = "../project_ev/user_profile.md"  # Adjust if running from different directory

# Try to find the file
import os
for possible_path in [
    "user_profile.md",
    "../project_ev/user_profile.md",
    "../../project_ev/user_profile.md",
    Path.cwd() / "user_profile.md",
    Path.home() / "OneDrive/Documentatie - professioneel - opleiding/AI pro 2025-26/Project - Gen AI/project_ev/user_profile.md"
]:
    if Path(possible_path).exists():
        profile_path = possible_path
        print(f"Found profile at: {profile_path}")
        break

# Initialize RAG manager
rag_manager = RAGUserProfileManager(profile_path=profile_path)
rag_manager.load_profile()
rag_manager.index_profile()

# Create RAG-enabled pipeline
rag_pipeline = LLMPipelineWithRAG(
    rag_manager=rag_manager,
    ollama_base_url="http://localhost:11434",
    ollama_model="llama3:8b",
    backend="auto",
    home_location="Kortrijk"
)

Found profile at: user_profile.md
Loaded 33 profile chunks.
Indexed 33 chunks into ChromaDB.


In [57]:
# Test RAG retrieval for sample queries
test_queries = [
    "schedule my trips this week as a normal workweek",
    "schedule next friday as a normal workweek",
    "hobby drive to Waregem on Saturday at 10am, the hobby needs to be rescheduled, the time of the hobby is same as normal",
    "Trip to parents confirmed for next upcoming trip, schedule this",
    "weekend getaway"
]

print("=== RAG Context Retrieval Test ===\n")
for query in test_queries:
    context = rag_manager.retrieve_context(query, top_k=3)
    print(f"Query: {query}")
    print(f"Retrieved context:\n{context}...\n" if len(context) > 300 else f"Retrieved context:\n{context}\n")
    print("-" * 80)

=== RAG Context Retrieval Test ===

Query: schedule my trips this week as a normal workweek
Retrieved context:
Weekday commute: [Work from Mon-Fri and I drive from home to Arcelormittal Dunkerque, on Wednesday I only work half a day which is in the morning, single trip is 95km. On Tuesday evening I leave

Weekend commute: [On Sundays in the morning from 9:00 till 11:00 to Lille in France which is 50km single trip]

---

## Monthly Habits

Typical long trip pattern: [The last Sunday of the month we go to Antwerpen AUWERSSTRAAT to meet my parents, we typically leave around 13:00 and are back at 19:00]...

--------------------------------------------------------------------------------
Query: schedule next friday as a normal workweek
Retrieved context:
Work or away hours: [Typically I leave at 06:30 and back home at 19:00 except for Wednesdays back at 13:00]

1. Check the fixed month-day holidays first.
2. If the date is not fixed, calculate Easter Sunday for that year.
3. Then apply the 

In [58]:
# Test RAG pipeline with trip messages
rag_examples = [
    "schedule my trips this week as a normal workweek",
    "schedule next friday as a normal workweek",
    "hobby drive to Waregem on Saturday at 10am, the hobby needs to be rescheduled, the time of the hobby is same as normal",
    "Trip to parents confirmed for next upcoming trip, schedule this",
    "weekend getaway"
]

print("=== RAG-Enhanced Trip Parsing ===\n")
for msg in rag_examples:
    print(f"Message: {msg}")
    parsed, raw, status = rag_pipeline.parse_trip(msg, backend="auto")
    print(f"Status: {status}")
    print(f"Raw LLM output : {raw if raw else 'None'}...")
    print(f"Parsed result: {parsed}")
    print("-" * 80 + "\n")

=== RAG-Enhanced Trip Parsing ===

Message: schedule my trips this week as a normal workweek
Status: Parsed with Ollama
Raw LLM output : {
    "1": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-18",
        "from": "Kortrijk",
        "to": "Arcelormittal Dunkerque",
        "Time_leave": "06:00",
        "Time_arrival": null
    },
    "2": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-18",
        "from": "Arcelormittal Dunkerque",
        "to": "Kortrijk",
        "Time_leave": "12:00",
        "Time_arrival": null
    },
    "feedback_LLM": "All required information is present."
}...
Parsed result: {'1': {'action': 'add_trip', 'title': 'Work', 'date': '2026-05-18', 'from': 'Kortrijk', 'to': 'Arcelormittal Dunkerque', 'Time_leave': '06:00', 'Time_arrival': None}, '2': {'action': 'add_trip', 'title': 'Work', 'date': '2026-05-18', 'from': 'Arcelormittal Dunkerque', 'to': 'Kortrijk', 'Time_leave': '12:00', 'Time_ar

## How RAG Works Here

1. **Load Profile**: `RAGUserProfileManager` reads `user_profile.md` and splits it into semantic sections.
2. **Vector Database**: Each section is embedded and stored in ChromaDB using cosine similarity.
3. **Query Retrieval**: When a trip message comes in, the top-3 most relevant profile sections are retrieved.
4. **Prompt Injection**: The LLM receives both the user query AND the relevant profile context.
5. **Better Reasoning**: The model understands your habits, constraints, EV specs, and availability in real-time.

### Key Benefits
- Trip parsing is context-aware (knows your commute distance, charging window, habits)
- Handles ambiguous dates by understanding your weekly patterns
- Estimates missing distances based on your typical trips
- Respects your constraints (availability, charging preference, etc.)

### To Use in Production
```python
# Initialize once
rag_manager = RAGUserProfileManager(profile_path="path/to/user_profile.md")
rag_manager.load_profile()
rag_manager.index_profile()

# Create pipeline with RAG
rag_pipeline = LLMPipelineWithRAG(
    rag_manager=rag_manager,
    backend="ollama",  # or "transformers"
    home_location="Your City"
)

# Parse trip with automatic RAG context
parsed, raw, status = rag_pipeline.parse_trip("trip to Amsterdam tomorrow")

# COMPLETED UPDATES

✅ **RAGUserProfileManager Refactored with RecursiveCharacterTextSplitter**

- Replaced simple header-based splitting with `RecursiveCharacterTextSplitter` from `langchain_text_splitters`
- Chunks now preserve semantic boundaries with sliding-window overlap
- Successfully loaded user_profile.md into 33 overlapping chunks
- RAG retrieval system working with embeddings-based semantic search

TO DO

- improve RAG - it makes a bigger mess of the input given by the user, but user input is less precise.

 - ADDing to the llm judge to check input of the user and output of the model to keep this between guides & boundaries

- ADD sort of pipeline when simple sentence is given " schedule whole workweek" that the llm is repeating itself and gives out put for every day